<div style="width: 100%; height: 20px; background-color: red;"></div>
#!!!! This only works for dataset coded in the "v2" style with dynamic parameters in sidecars:
# EnCodecLatentDataset=EnCodecLatentDataset_dynamic_v2   #latents in "sidecar" npy and json files and have as many frames as the audio file
<div style="width: 100%; height: 20px; background-color: red;"></div>

In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

In [ ]:
# a little magic to get the output devices properly for both the rt synth and the html player
import os
os.environ["ALSA_CONFIG_PATH"] = "/usr/share/alsa/alsa.conf"
os.environ["ALSA_PLUGIN_DIR"]  = "/usr/lib/x86_64-linux-gnu/alsa-lib"

import sounddevice as sd # import *after* the os.environ settings
print("Default device:", sd.default.device)
print([d["name"] for d in sd.query_devices() if d["max_output_channels"]>0][-3:])

In [ ]:
import torch
import os
import numpy as np
from pathlib import Path
import time

import matplotlib.pyplot as plt
from IPython.display import Audio, display

from transformers import EncodecModel
from rnencodec.generator import RNNGeneratorSoft, EncodecRTPlayer

# for RNN4Control
from rnencodec.model.gru_audio_model import RNN, GRUModelConfig
from rnencodec.audioDataLoader.audio_dataset import LatentDatasetConfig
from rnencodec.audioDataLoader.audio_dataset import  efficient_codes_to_latents, preprocess_latents_for_RNN # , latents_to_audio_simple,

from rnencodec.utils.utils import load_sidecar #import load_sidecar_normalized
from rnencodec.utils.utils import plot_audio_with_params_two_yaxes  #for nice audio + param plots

from rnencodec.utils.utils import read_ecdc_reconstruct_audio, interpolate_breakpoints
from rnencodec.utils.utils import plot_audio

# system params
sr=24000
frame_rate=75
device='cpu' #best for inference
buffersize = 320 #[NOTE - not tested on values other than 24000/75 - the frame length of encodec codes in samples]

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>"User" Synth/Encodec Parameters</b>

In [ ]:
g_hopsize=8  # This has a big effect on the off-line generate() because the smaller it is, the more old fashioned context switching time we spend
g_chunksize=24

offline_render_duration=5 #seconds
offline_render_frames=offline_render_duration*75

profiles=["syntex1", "syntex7", "nsynth2", "water", "ddsp_violin", "babble", "nsynth7", "engineC"]
synthprofile=profiles[3] # <<<<<===============================

# if you trained on discrete values, you can force the widget parameters to be discrete, too. See the "nsynth2" profile for pitch. 
# You could also force your "one hot" values to be either 0 or 1 this way, too (though it is more fun to use continuous values for these)
g_desc_param_vals=None # by default - changed in some profiles below


if synthprofile=="syntex1" : 
    g_param_labels = ["Pitch", "Amp"]
    g_norm_param_vals = [.5, .5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:1] # the first n will be passed to the RNN generator

elif synthprofile=="syntex7" :
    #--g_param_labels = ["c1", "c2", "c3", "c4", "c5", "c6", "c7", "param 1", "Amp"]
    g_param_labels = ['param 1', 'Chirps 1H', 'Applause 1H', 'Bugs 1H', 'Peepers 1H', 'Pistons 1H', 'Wind 1H', 'TokWotal 1H'  ]
    
    #--g_norm_param_vals = [1,  0,     0,    0,    0,    0,    0,    0.5]  #for the synth whose paramters can be different than those it sends to the NN
    g_norm_param_vals = [.95,  0,     0,    1,    0,    0,    0,    0]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:8] # the first n will be passed to the RNN generator

    #run_directory = str(Path('../output_keep/20250924_144943_multiclass_test'))
    #run_directory = str(Path('output/20251024_183850_multiclass_test_softcascade'))
    #run_directory = str(Path('output/20251025_161452_multiclass_test_HARDcascade'))
    #run_directory = str(Path('output/20251027_181934_multiclass_test_SOFTcascade'))
    
    #run_directory = str(Path('output/20251128_140835_multiclass_test_SOFTcascade'))

    run_directory = str(Path('output/20260122_134328_datapreptest'))
    
    checkpoint_fname =   "last_checkpoint.pt" #"checkpoint_125.pt" # "checkpoint_25.pt" # "checkpoint_25.pt" #

    #For comoparing RNN param-driven synthesis to a dataset parameterization
    # some to try: 094, 027, 082, 004, 293, 059, 058, 093
    test_file_path = "/slowdisk/data/syn7_v4/hf_dataset/tokens/validation"

    #["ChirpPattern", "DSApplause", "DSBugs", "DSPeepers", "DSPistons", "DSWind", "TokWotalDuet"]
    #sound_name="ChirpPattern--irreg_exp-00.20--c-01--x-99"
    sound_name="DSApplause--numClappers_exp-01.00--c-02--x-98"
    #sound_name="DSBugs--busybodyFreqFactor-00.95--c-01--x-98"
    #--sound_name="DSPeepers--max_range-00.45--c-00--x-98"
    test_fname = test_file_path  + "/" + sound_name     # shape: (T, 2), dtype float16 or float32
    metapath="/slowdisk/data/syn7_v4/hf_dataset/conditioning_config.json" # just one meta file for all sidecars



elif synthprofile=="nsynth2" : #nsynth
    g_param_labels = ["class", "pitch", "amp"]
    g_norm_param_vals = [0,       .5,     .5 ]  # for the synth whose paramters can be different than those it sends to the NN
    g_desc_param_vals = [0,       13,      0 ]  # 0 for float vals, n for n discrete values including endpoints
    g_init_cond=g_norm_param_vals[:]            # for the neural network
    run_directory = str(Path('output/20251024_140730_nsynth_soft_cascade'))
    checkpoint_fname =   "last_checkpoint.pt"
    
elif synthprofile=="water" :
    g_param_labels = ["pos"]
    g_norm_param_vals = [.5 ]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    #run_directory = str(Path('output/20251024_124629_quickstart_softcascade'))
    #run_directory = str(Path('output/20251120_135045_water_0'))
    #run_directory = str(Path('output/20260128_231114_water_soft.newenvtest'))

    run_directory = str(Path("../quickstart/output_26.02.03.15.49")) # Path to save the trained model and configs
    checkpoint_fname = "last_checkpoint.pt"   # Checkpoint to use for inference (e.g., "last_checkpoint.pt" or "checkpoint_75.pt")


    test_file_path = "../data/quickstartdata/waterfill/tokens/test"
    sound_name="ZOOM0028"
    test_fname = test_file_path  + "/" + sound_name     # shape: (T, 2), dtype float16 or float32
    metapath=None


elif synthprofile=="ddsp_violin" :
    g_param_labels = ["pitch", "amp"]
    g_norm_param_vals = [.5, .7 ]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    run_directory = str(Path('output/20251017_113920_ddsp_violin_seql_75_corrected'))
    checkpoint_fname =   "last_checkpoint.pt" # "checkpoint_25.pt" # "checkpoint_25.pt" #


elif synthprofile=="babble" :
    g_param_labels = ["sex"]
    g_norm_param_vals = [.5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    run_directory = str(Path('output/20251119_163720_babble_gender_mf'))
    checkpoint_fname =   "checkpoint_425.pt" # "last_checkpoint.pt" # "checkpoint_25.pt" # 


elif synthprofile=="nsynth7" :
    g_param_labels = ["pitch", "velocity", "003clarinet", "006frenchhorn","094bass", "162flute","272voice-ee","295voice-oo","887organ"]
    g_norm_param_vals = [.5,.4,1,0,0,0,0,0,0]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    run_directory = str(Path('output/20251128_111405_nsynth7_trial2'))
    checkpoint_fname =   "checkpoint_175.pt" # "last_checkpoint.pt" # "checkpoint_425.pt" #  "checkpoint_25.pt" # 

elif synthprofile=="engineC" :
    g_param_labels = ["RPM", "Torque"]
    g_norm_param_vals = [.5,.5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    #run_directory = str(Path('output/20260109_175027_engineC_trial'))
    #run_directory = str(Path('output/20260112_153827_engineC_SOFTtrial'))
    #run_directory = str(Path('output/20260113_144103_engineC_SOFTtrial'))
    #run_directory = str(Path('output/20260113_155725_engineC_SOFTtrial_TFONLY'))
    #run_directory = str(Path('output/20260113_170647_engineC_SOFTtrial_TFNOTUSED'))

    #run_directory = str(Path('output/20260113_180228_engineC_SOFTtrial_TFCYCLE_TAU.25'))
    #---run_directory = str(Path('output/20260115_144033_engineC_SOFTtrial_TFCYCLE_TAU.35_topnsoft'))

    #run_directory = str(Path('output/20260123_114246_engineC_SOFTtrial_TFCYCLE_TAU.35_topnsoft_newprep'))
    run_directory = str(Path('output/20260124_161336_engineC_SOFTtrial_TFCYCLE_TAU.35_topnsoft_postmerge'))
    checkpoint_fname =  "last_checkpoint.pt" #  "checkpoint_100.pt" # "last_checkpoint.pt" #"checkpoint_75.pt" # "last_checkpoint.pt" # "checkpoint_175.pt" #  "checkpoint_425.pt" #  "checkpoint_25.pt" # 


    
    #For comoparing RNN param-driven synthesis to a dataset parameterization
    # some to try: 094, 027, 082, 004, 293, 059, 058, 093

    
    #---test_file_path = "/home/lonce/slowdisk/data/ProceduralEngineSounds/ECDC/C_full_set_ecdc_HF/tokens"
    test_file_path = "/slowdisk/data/ProceduralEngineSounds/CSV/proceduralengine_prepped/hf_dataset/tokens/test"
    sound_name="094_Engine-C"
    test_fname = test_file_path  + "/" + sound_name     # shape: (T, 2), dtype float16 or float32
    metapath=None
    

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>RNN Parameters</b>

In [ ]:
#Used to alter the config after it is loaded from the checkpoint
g_top_n = 3 #'Sample from the top N most likely outputs.'
g_temperature = .8 #'Controls the randomness of predictions.'
g_sample_mode="sample"

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>loading</b>

In [ ]:
# load the encoder model 
#####################################################################
enc_model = EncodecModel.from_pretrained("facebook/encodec_24khz")
enc_model.eval()
enc_model.device

# load the RNN model 
#####################################################################
config_path = os.path.join(run_directory, "config_v2.pt")
checkpoint_path = os.path.join(run_directory, "checkpoints", checkpoint_fname) 

assert os.path.exists(run_directory), f"Run directory not found: {run_directory}"
assert os.path.exists(config_path), f"Config file not found: {config_path}"
assert os.path.exists(checkpoint_path), f"Checkpoint file not found: {checkpoint_path}"

saved_configs = torch.load(config_path, weights_only=False)
model_config = GRUModelConfig(**saved_configs["model_config"])
data_config = LatentDatasetConfig(**saved_configs["data_config"])

# could be different from config defaults or what we trained with
# model_config.cascade='hard' # 
# model_config.hard_sample_mode=g_sample_mode
# model_config.top_n_hard=g_top_n
# model_config.temperature_hard=g_temperature

rnngen = RNNGeneratorSoft.from_checkpoint(checkpoint_path, model_config, data_config, enc_model, g_chunksize, g_hopsize, sample_mode_outside=g_sample_mode, top_k_outside=g_top_n, temperature_outside =g_temperature)
print("Model successfully loaded from checkpoint.")
print(f"Using device = {device}") 
if getattr(model_config, 'cascade') == 'hard' :
    print(f"!!!!  OI, You are using hard-sampling in the saved model. The soft parameters (eg g_top_n) will be ignored. The model confic is: {model_config}")

In [ ]:
model_config

In [ ]:
data_config

In [ ]:
# # Visualize hops shiting into the "right" side of chunks
# print(f"rnngen.codebuf is {rnngen.codebuf}")
# newfoo = rnngen.getNextCodeChunk(g_init_cond, g_hopsize)
# print(f"rnngen.codebuf is {rnngen.codebuf}")

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b style="font-size: 20px;">RNNGenerator testing -  the intended way to generate "off line" </b>

In [ ]:
#++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#  NOW with RNNGenerator Directly 
#++++++++++++++++++++++++++++++++++++++++++++++++++
bazrnn = RNNGeneratorSoft.from_checkpoint(checkpoint_path, model_config, data_config, enc_model, g_chunksize, g_hopsize, sample_mode_outside=g_sample_mode, top_k_outside=g_top_n, temperature_outside =g_temperature)
#warm up prevents noise bursts at the begininng of the audio run
_ = bazrnn.warmup(g_init_cond, 10)

# Note that the param array is used for the whole hop. 
# Update the param array per hop for dynamic control 

segments = []
numhops=int(offline_render_frames/g_hopsize)
print(f'numhops will be {numhops}')
#params = np.linspace(0, 1, numhops, dtype=np.float32)[:, np.newaxis]


start = time.perf_counter()
for i in range(numhops):
    p = torch.tensor([g_init_cond], dtype=torch.float32, device=device)
    params_seq = p.view(1, -1).expand(g_hopsize, -1)  # shape (T, 2), view with stride 0
    #print(f' for hop {i}, params_seq is {params_seq}')
    a=bazrnn.getNextAudioHop(params_seq)  # returns a 1D float32 array like in your example
    segments.append(a)
fullaudio = np.concatenate(segments, dtype=np.float32)  # one allocation

elapsed = time.perf_counter() - start

print(f'Total time to render {fullaudio.shape[0]/sr} secs: {elapsed}')
print(f"Generated { fullaudio.shape[0]} samples at {sr} Hz in {elapsed:.2f}s.")


In [ ]:
display(Audio(fullaudio, rate=sr)) 

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Synthesize offline from Parameters</b>

In [ ]:
test_fname
Path(test_fname)

In [ ]:


whole_param_seq = load_sidecar(test_fname) # tensor of shape (T, D)


trialsegments=[]
numhops= whole_param_seq.shape[0] // g_hopsize
print(f' will run for {numhops} hops')

for i in range(numhops):
    p = torch.tensor([g_init_cond], dtype=torch.float32, device=device)
    params_seq = whole_param_seq[i*g_hopsize : i*g_hopsize + g_hopsize] # p.view(1, -1).expand(g_hopsize, -1)  # shape (T, 2), view with stride 0
    #print(f' for hop {i}, params_seq is {params_seq}')
    a=bazrnn.getNextAudioHop(params_seq)  # returns a 1D float32 array like in your example
    trialsegments.append(a)
trialaudio = np.concatenate(trialsegments, dtype=np.float32)  # one allocation


In [ ]:
whole_param_seq[6:10]

In [ ]:
plot_audio_with_params_two_yaxes(
    trialaudio,
    whole_param_seq,
    audio_sr=24000,
    param_sr=75,
    param_names=g_param_labels, 
    subtitle=sound_name
)

display(Audio(trialaudio, rate=sr)) 

print(f' Compare to .ecdc data reconstructed:' )

data_audio_from_ecdc, _sr = read_ecdc_reconstruct_audio(test_fname+'.ecdc')

plot_audio(data_audio_from_ecdc, title="Encodcec data->audio", subtitle=sound_name)
display(Audio(data_audio_from_ecdc, rate=sr))     

In [ ]:
#if you call with hopsizes larger than the chunk you set up, you'll get them, but it won't take advantage of the chunking strategy)

In [ ]:
g_init_cond

In [ ]:
#foo = bazrnn.warmup([.5,.5, .5], 1000)  
#foo=bazrnn.warmup([0, 0, 0, 0, 0, 0, 1, 0.5], 1000)
foo=bazrnn.warmup(g_init_cond, 10)
display(Audio(foo, rate=sr)) 

In [ ]:
params_seq

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Build your own sequence</b>

In [ ]:
byo_param_seq = interpolate_breakpoints([[(0,0), (2, 1), (4, 0), (6, 1), (8, 0), (10, 1), (12, 0), (14,1)]], 75) # load_sidecar(test_fname) # tensor of shape (T, D)

byo_trialsegments=[]
numhops= byo_param_seq.shape[0] // g_hopsize
print(f' will run for {numhops} hops')

start = time.perf_counter()
for i in range(numhops):
    params_seq = byo_param_seq[i*g_hopsize : i*g_hopsize + g_hopsize] # p.view(1, -1).expand(g_hopsize, -1)  # shape (T, 2), view with stride 0
    #print(f' for hop {i}, params_seq is {params_seq}')
    a=bazrnn.getNextAudioHop(params_seq)  # returns a 1D float32 array like in your example
    byo_trialsegments.append(a)
byo_audio = np.concatenate(byo_trialsegments, dtype=np.float32)  # one allocation

elapsed = time.perf_counter() - start
print(f'Total time to render {byo_audio.shape[0]/sr} secs: {elapsed}')


In [ ]:
plot_audio_with_params_two_yaxes(
    byo_audio,
    byo_param_seq,
    audio_sr=24000,
    param_sr=75,
    param_names=g_param_labels, 
    subtitle=sound_name
)

display(Audio(byo_audio, rate=sr)) 